# 00 — Kurulum ve Erişim Kontrolü

Bu notebook'un amacı, asıl analize başlamadan önce **her şeyin çalıştığından emin olmak**:

1. Gerekli Python paketlerini kurmak,
2. Google Drive'ı bağlayıp proje klasörünü bulmak,
3. Google Earth Engine'e (GEE) giriş yapmak,
4. 7 şehrin sınırlarını haritada göstermek,
5. Kullanacağımız tüm uydu/iklim veri setlerinden küçük birer örnek okuyup erişim olup olmadığını tablo halinde görmek.

**Google Earth Engine nedir?** Google'ın, petabaytlarca uydu görüntüsünü (Landsat, MODIS, ERA5 vb.) kendi sunucularında tutan ve bu veriler üzerinde hesaplamayı yine kendi sunucularında yapan ücretsiz bir platformdur. Biz görüntüleri bilgisayarımıza indirmeyiz; "şu bölgede şu hesabı yap, sonucu bana gönder" deriz. Bu yüzden büyük veriyle bile Colab'da çalışabiliyoruz.

**Kota notu:** Ticari olmayan (noncommercial) projelerin aylık ücretsiz işlem kotası var. Bu notebook çok küçük sorgular yapar; kotayı neredeyse hiç harcamaz.

## Ön koşul (bir kez yapılır)
1. GEE kaydı yapılmış olmalı (Project ID `config.py` içinde: `yapay-zeka-proje-2026`).
2. Colab'da sol menüdeki **anahtar (Secrets)** simgesine `GITHUB_TOKEN` adıyla GitHub erişim anahtarı eklenmiş ve **Not defteri erişimi** açılmış olmalı (README → "Colab kurulumu").

## 1. Paket kurulumu
Colab'da çoğu paket hazır gelir; sadece eksik olanları kuruyoruz. `-q` sessiz kurulum demektir.

In [ ]:
!pip install -q earthengine-api geemap pymannkendall

## 2. Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Repo herkese açık olduğu için **token gerekmez**. Colab'ın **Gizli anahtarlar (Secrets)** bölümünde `GITHUB_TOKEN` tanımlıysa yine kullanılır (repo gizli yapılırsa gerekir); token ekrana yazdırılmaz.

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
GITHUB_BRANCH = "main"   # kendi dalınızdaki kodu çalıştırmak için dal adını yazın (ör. "amine/geoclimate-uyum")
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        mesaj = r.stderr or r.stdout
        raise RuntimeError("git hatası: " + (mesaj.replace(token, "***") if token else mesaj))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    # Repo herkese açık: token gerekmez. Colab Secrets'ta GITHUB_TOKEN varsa kullanılır (repo gizli yapılırsa gerekir).
    try:
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        token = ""
    adres = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
    if not os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["clone", "-q", adres, CODE_DIR], token)
    # Seçilen dalın en güncel hâline geç (Colab kopyasındaki yerel değişiklikler atılır)
    _git(["-C", CODE_DIR, "fetch", "-q", "origin", GITHUB_BRANCH], token)
    _git(["-C", CODE_DIR, "checkout", "-q", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| dal:", GITHUB_BRANCH if IN_COLAB else "yerel", "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

## 3. Google Earth Engine'e giriş
İlk çalıştırmada bir bağlantı/pencere açılır: Google hesabınızı seçip izin verin. Sonra kimlik bilgisi Colab oturumunda saklanır.

Project ID'yi `config.py`'a yazmadıysanız aşağıdaki satırda `None` yerine tırnak içinde yazabilirsiniz, ör. `GEE_PROJECT = "yapay-zeka-proje-123456"`.

In [ ]:
import ee
GEE_PROJECT = None   # boş bırakılırsa config.GEE_PROJECT kullanılır

from src import gee_utils as gu
importlib.reload(gu)
from src.gee_utils import init_ee, get_city_boundaries, list_turkey_provinces
init_ee(GEE_PROJECT)

## 4. Şehir sınırları
Şehir (il) sınırlarını **FAO GAUL 2015** veri setinden alıyoruz. GAUL, BM Gıda ve Tarım Örgütü'nün yayımladığı idari sınır veritabanıdır; "level1" Türkiye için il düzeyidir.

GAUL'da il adları Türkçe karakter olmadan yazılır (ör. "Sanliurfa"). Aşağıdaki hücre 7 şehrin hepsinin bulunduğunu kontrol eder; bulunamayan olursa GAUL'daki tüm il adlarını yazdırır, biz de `config.py`'daki adı düzeltiriz.

In [ ]:
cities_fc = get_city_boundaries()
found = sorted(cities_fc.aggregate_array("ADM1_NAME").getInfo())
expected = sorted(config.CITIES.values())
missing = [c for c in expected if c not in found]

print("Bulunan şehirler :", found)
if missing:
    print("BULUNAMAYAN      :", missing)
    print("\nGAUL'daki tüm Türkiye il adları:")
    print(list_turkey_provinces())
else:
    print("7 şehrin hepsi bulundu.")

# Her ilin alanı (km²) — çalışma alanlarının büyüklüğünü görmek için
areas = cities_fc.map(lambda f: f.set("area_km2", f.geometry().area(1000).divide(1e6)))
import pandas as pd
area_df = pd.DataFrame({
    "city": areas.aggregate_array("ADM1_NAME").getInfo(),
    "area_km2": areas.aggregate_array("area_km2").getInfo(),
}).round(0).sort_values("area_km2", ascending=False)
area_df

### Etkileşimli harita
Aşağıda 7 ilin sınırları kırmızı çizgiyle, Esri uydu görüntüsü üzerinde görünmeli. Sağ üstteki katman düğmesinden Google Hybrid arka planına geçebilirsiniz. Haritayı yakınlaştırıp kaydırabilirsiniz. Harita ayrıca `results/figures/sehir_sinirlari.html` olarak kaydedilir.

In [ ]:
m = gu.make_map(center=(39.0, 35.0), zoom=6)
m.addLayer(cities_fc.style(color="red", fillColor="ff000022", width=2), {}, "Çalışma alanları")
m.add_layer_control()
m.to_html(os.path.join(config.FIGURES_DIR, "sehir_sinirlari.html"))
m

### Rapor için statik harita (PNG)
Raporda etkileşimli harita kullanılamayacağı için aynı sınırları sabit bir görsel olarak da kaydediyoruz. Sınırlar indirilmeden önce 500 m hassasiyetle sadeleştirilir (dosya küçük kalsın diye).

In [ ]:
import geemap
import matplotlib.pyplot as plt

gdf = geemap.ee_to_gdf(cities_fc.map(lambda f: f.simplify(500)).select(["ADM1_NAME"]))
fig, ax = plt.subplots(figsize=(11, 5))
gdf.plot(ax=ax, column="ADM1_NAME", cmap="tab10", edgecolor="black", alpha=0.7)
for _, row in gdf.iterrows():
    p = row.geometry.representative_point()
    ax.annotate(row["ADM1_NAME"], (p.x, p.y), ha="center", fontsize=9, weight="bold")
ax.set_title("Çalışma alanları (FAO GAUL 2015 il sınırları)")
ax.set_xlabel("Boylam"); ax.set_ylabel("Enlem")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "sehir_sinirlari.png"), dpi=200)
plt.show()

## 5. Veri setlerine erişim kontrolü
Her veri setinden **tek bir noktada (Konya merkezi)** çok küçük bir değer okuyoruz. Amaç veriyi analiz etmek değil, sadece "erişebiliyor muyuz, beklediğimiz bantlar ve tarihler var mı" sorusunu cevaplamak.

Ayrıca bilinen veri kısıtlarını da burada doğruluyoruz:
- Landsat 9'un ilk görüntü tarihi (2021 sonu olmalı),
- JRC su verisinin son yılı (2021 olmalı),
- WorldCover v200'ün yılı (2021 olmalı).

In [ ]:
C = config.COLLECTIONS
pt = ee.Geometry.Point([32.49, 37.87])   # Konya merkez
reg = pt.buffer(5000)

def first_date(ic, ascending=True):
    return ee.Date(ic.sort("system:time_start", ascending).first().get("system:time_start")).format("YYYY-MM-dd")

checks = {
    "Landsat 8 (C2 L2)": (C["landsat8"], lambda: (
        f"2024 yazı görüntü sayısı: {ee.ImageCollection(C['landsat8']).filterBounds(pt).filterDate('2024-06-01','2024-09-01').size().getInfo()}, "
        f"2014 yazı: {ee.ImageCollection(C['landsat8']).filterBounds(pt).filterDate('2014-06-01','2014-09-01').size().getInfo()}")),
    "Landsat 9 (C2 L2)": (C["landsat9"], lambda: (
        f"ilk görüntü: {first_date(ee.ImageCollection(C['landsat9']).filterBounds(pt)).getInfo()}, "
        f"2024 yazı görüntü sayısı: {ee.ImageCollection(C['landsat9']).filterBounds(pt).filterDate('2024-06-01','2024-09-01').size().getInfo()}")),
    "ESA WorldCover v200": (C["worldcover"], lambda: (
        f"yıl: {first_date(ee.ImageCollection(C['worldcover'])).getInfo()[:4]}, "
        f"Konya merkez sınıf kodu: {ee.ImageCollection(C['worldcover']).first().reduceRegion(ee.Reducer.mode(), pt, 10).get('Map').getInfo()}")),
    "MODIS LST (MOD11A2)": (C["modis_lst"], lambda: (
        f"ilk: {first_date(ee.ImageCollection(C['modis_lst'])).getInfo()}, "
        f"son: {first_date(ee.ImageCollection(C['modis_lst']), False).getInfo()}, "
        "Temmuz 2024 gündüz LST (°C): "
        f"{round(ee.ImageCollection(C['modis_lst']).filterDate('2024-07-01','2024-08-01').select('LST_Day_1km').mean().multiply(0.02).subtract(273.15).reduceRegion(ee.Reducer.mean(), reg, 1000).get('LST_Day_1km').getInfo(), 1)}")),
    "ERA5-Land aylık (MONTHLY_AGGR)": (C["era5_land"], lambda: (
        lambda d: f"Temmuz 2024 toprak nemi (m³/m³): {round(d['volumetric_soil_water_layer_1'], 3)}, 2 m sıcaklık (°C): {round(d['temperature_2m'] - 273.15, 1)}")(
        ee.ImageCollection(C['era5_land']).filterDate('2024-07-01','2024-08-01').first()
          .select(config.ERA5_BANDS).reduceRegion(ee.Reducer.mean(), pt, 11132).getInfo())),
    "JRC Global Surface Water (YearlyHistory)": (C["jrc_water"], lambda: (
        f"ilk yıl: {first_date(ee.ImageCollection(C['jrc_water'])).getInfo()[:4]}, "
        f"son yıl: {first_date(ee.ImageCollection(C['jrc_water']), False).getInfo()[:4]}")),
    "CHIRPS yağış (PENTAD)": (C["chirps"], lambda: (
        f"2024 içindeki 5 günlük görüntü sayısı: {ee.ImageCollection(C['chirps']).filterDate('2024-01-01','2025-01-01').size().getInfo()}, "
        f"son: {first_date(ee.ImageCollection(C['chirps']), False).getInfo()}")),
    "SRTM rakım": (C["dem"], lambda: (
        f"Konya merkez rakım (m): {ee.Image(C['dem']).reduceRegion(ee.Reducer.first(), pt, 30).get('elevation').getInfo()}")),
}

rows = []
for name, (gee_id, fn) in checks.items():
    try:
        detail = fn()
        status = "Erişildi"
    except Exception as e:
        detail, status = str(e)[:200], "ERİŞİLEMEDİ"
    rows.append({"veri": name, "gee_id": gee_id, "durum": status, "detay": detail})
    print(f"[{status}] {name}: {detail}")

access_df = pd.DataFrame(rows)
access_df.to_csv(os.path.join(config.TABLES_DIR, "erisim_kontrolu.csv"), index=False, encoding="utf-8-sig")
access_df

### Bu tabloyu nasıl yorumlarız?
- **durum** sütununun tamamı "Erişildi" olmalı. "ERİŞİLEMEDİ" olan satır varsa **detay** sütunundaki hata mesajını bize iletin.
- **Landsat 9 ilk görüntü** 2021 sonbaharı civarında olmalı → bu yüzden 2014 kompoziti sadece Landsat 8'den yapılacak.
- **JRC son yıl** 2021 olmalı → 2024 su alanını kendimiz Landsat MNDWI ile hesaplayacağız.
- **WorldCover yılı** 2021 olmalı → eğitim etiketlerimizin yılı.
- MODIS Temmuz LST'si ve ERA5 2 m sıcaklığı makul değerler (Konya yazı için kabaca 25–45 °C arası) vermeli; bu, ölçek faktörlerini doğru uyguladığımızı gösterir.

## Faz 0 bitti mi?
- [ ] 7 şehrin sınırları haritada görünüyor (`results/figures/sehir_sinirlari.html` ve `.png`)
- [ ] `results/tables/erisim_kontrolu.csv` içinde tüm veri setleri "Erişildi"